# Single-goal radial ranking by legal path distance

Run on a Colab GPU. The goal is fixed at **(15, 15)** and the wall layout is **(32, 30)**. This side experiment uses hand-positioned ball images and the frozen I-JEPA encoder.

- **Euclidean distance selects the band.** The bands are 6–12, 12–20, 20–28, 28–34, 34–42, 42–50, 50–58, and 58–64. The last band reaches the farthest corner, 63.64 units from the goal. We place samples on two radii per band at 15° intervals, keeping valid positions only.
- **Shortest legal path distance labels the pair.** The path can go straight wherever the solid wall permits and can bend around the doorway corners. We rank two observations if the legal path distances to the goal differ by at least 0.5 units. Within-band and across-band pairs are both eligible. At the same Euclidean radius, the points can still differ in legal path distance.
- **Fit cumulatively.** Start from identity, keep all earlier pairs eligible, and warm-start each band with the previous weights. Each band uses 500 ranking updates.
- **Measure generalization.** Use a separate grid and 7.5°-shifted rays. Report held-out ordering inside and beyond the current Euclidean radius, in the current band, across positions on the opposite side, and within each same-radius ring. The heatmaps show the full map after every stage.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
RESUME_RUN_ID=None
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/geodesic_radial_v2'/RUN_ID
REPO=Path('/content/i-jepa-wm-geodesic-radial')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='421e8ed16466d33a721b138205a19e412282df91'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ARTIFACTS=Path('/content/geodesic-radial-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Run ID:',RUN_ID)
print('Output:',OUTPUT)

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in [(ENCODER_DRIVE,ENCODER,ENCODER_SHA)]:
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

## Fit the eight checkpoints

On resume, set `RESUME_RUN_ID` above to the printed ID and use the same numerical runtime. Completed checkpoints are reused after the frames are encoded again.

In [ ]:
started=time.monotonic()
shell([sys.executable,'-m','src.mylonas_ijepa_wm.geodesic_radial',
       '--output-dir',OUTPUT,'--encoder-checkpoint',ENCODER,
       '--dino-wm-root',DINO_ROOT,'--device','cuda'],cwd=REPO)
print('Run minutes:',round((time.monotonic()-started)/60,2))
report=json.loads((OUTPUT/'comparison.json').read_text())

## Check pair coverage and held-out ranking

Accuracy is the percentage of eligible pairs for which the farther point by legal path distance has higher latent MSE. The same-radius test compares points within each held-out ring. The opposite-room test compares positions in the right room against one another, ranking them relative to the left-room goal.

In [ ]:
import pandas as pd

stages=report['stages']
display(pd.DataFrame([
    dict(band=s['band'],outer_radius=s['outer_radius'],training_nodes=len(s['training_points']),**s['pair_counts'])
    for s in stages
]).set_index('band'))
rows=pd.DataFrame([
    dict(band=s['band'],cohort=cohort,pairs=results['raw']['pairs'],
         raw_pct=None if results['raw']['correct'] is None else 100*results['raw']['correct'],
         adapted_pct=None if results['adapted']['correct'] is None else 100*results['adapted']['correct'])
    for s in stages for cohort,results in s['metrics'].items()
])
display(rows.round(2))
rows.to_csv(OUTPUT/'ranking_metrics.csv',index=False)

## View the full map

The legal-distance panel uses simulation units. Each latent MSE panel has its own unclipped color scale. The star marks the goal, open white circles mark fitted positions, dashed arcs mark Euclidean band limits, and gray denotes the solid wall. The y-axis increases downward as in the simulator image.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
import numpy as np

d=report['design']
points=np.asarray(d['points'])
axis=d['axis']
panels=[('Legal path distance',report['truth'],None),('Raw latent MSE',report['raw'],None)]
panels.extend((f"Band {s['band']} · R={s['outer_radius']}",s['adapted'],s) for s in stages)
cols=3
rows=(len(panels)+cols-1)//cols
fig,axes=plt.subplots(rows,cols,figsize=(14,4.5*rows),constrained_layout=True)
for ax in axes.flat[len(panels):]:
    ax.set_visible(False)
for ax,(title,values,stage) in zip(axes.flat,panels):
    field=np.asarray([np.nan if i is None else values[i] for i in d['grid']]).reshape(len(axis),len(axis))
    img=ax.pcolormesh(axis,axis,field,cmap='viridis',shading='nearest',vmin=0)
    ax.scatter(*d['goal'],marker='*',c='black',edgecolor='white',s=135,zorder=4)
    for top,bottom in [(axis[0],26),(34,axis[-1])]:
        ax.add_patch(Rectangle((29,top),6,bottom-top,facecolor='#424242'))
    if stage:
        ax.scatter(*points[stage['training_points']].T,s=10,facecolors='none',edgecolors='white',linewidths=.6)
        ax.add_patch(Circle(d['goal'],stage['outer_radius'],fill=False,color='white',ls='--',lw=.9))
    ax.set(title=title,xlim=(axis[0],axis[-1]),ylim=(axis[-1],axis[0]),
           aspect='equal',xlabel='Ball x',ylabel='Ball y')
    fig.colorbar(img,ax=ax,shrink=.78,label='Simulation units' if title.startswith('Legal') else 'Latent MSE')
fig.savefig(OUTPUT/'full_map_heatmaps.png',dpi=150)
plt.show()
plt.close(fig)